# Timestomping Detection Tool - Feature Engineering

This notebook extracts forensic features from the merged dataset for timestomping detection.

## Input

- `data_merged.csv` from notebook 01

## Features Extracted (31 total)

- **Forensic patterns (15)**: Zero nanoseconds, time reversal, timestamp modifications
- **Cross-artifact validation (3)**: LogFile evidence, UsnJrnl evidence, validation score
- **File characteristics (9)**: File types, path depth, filename length, location
- **Timestamp relationships (2)**: Creation/modified equality, timestamp patterns
- **Refined detection (2)**: Combined detection patterns

## Output

- `data_features.csv` - Dataset with 31 extracted features ready for detection


## Cell 1: Imports and Setup

In [1]:
# Cell 1: Imports and Setup

import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Libraries imported successfully
Pandas version: 2.3.2
NumPy version: 2.3.3


## Cell 2: User Configuration

**EDIT THIS SECTION** if you changed paths in notebook 01

In [2]:
# Cell 2: User Configuration

# Input file (output from notebook 01)
INPUT_DIR = "/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Post-Processing Filter/12-KSK"
INPUT_FILE = "data_merged.csv"

# Output directory
OUTPUT_DIR = INPUT_DIR
OUTPUT_FILE = "data_features.csv"

# Construct paths
input_path = os.path.join(INPUT_DIR, INPUT_FILE)
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)

print("Configuration loaded")
print("-" * 80)
print(f"Input file: {input_path}")
print(f"  Exists: {os.path.exists(input_path)}")
print(f"Output file: {output_path}")
print("-" * 80)

if not os.path.exists(input_path):
    raise FileNotFoundError(f"Merged data not found: {input_path}")


Configuration loaded
--------------------------------------------------------------------------------
Input file: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Post-Processing Filter/12-KSK/data_merged.csv
  Exists: True
Output file: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Post-Processing Filter/12-KSK/data_features.csv
--------------------------------------------------------------------------------


## Cell 3: Load Merged Dataset

In [3]:
# Cell 3: Load Merged Dataset

print("=" * 80)
print("LOADING MERGED DATASET")
print("=" * 80)

# Load merged data
df = pd.read_csv(input_path, low_memory=False)

print(f"\nDataset loaded successfully")
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

# Display column names
print(f"\nAvailable columns:")
print(df.columns.tolist())


LOADING MERGED DATASET

Dataset loaded successfully
  Records: 3,494
  Columns: 42
  Memory usage: 4.70 MB

Available columns:
['merge_key', 'lf_lsn', 'EventTime(UTC+8)', 'lf_event', 'lf_detail', 'File/Directory Name', 'lf_full_path', 'lf_creation_time', 'lf_modified_time', 'lf_mft_modified_time', 'lf_accessed_time', 'Redo', 'Target VCN', 'Cluster Index', 'suspicious_lsn_usn_lf', 'usn_event_time', 'usn_usn', 'usn_filename', 'usn_full_path', 'usn_event_info', 'SourceInfo', 'FileAttribute', 'Carving Flag', 'usn_file_reference_number', 'usn_parent_file_reference_number', 'suspicious_lsn_usn_usn', 'suspicious_category', 'suspicious_detail', 'suspicious_source', 'lf_event_count', 'usn_event_count', 'has_logfile_evidence', 'has_usnjrnl_evidence', 'cross_artifact_validation_score', 'has_suspicious_label', 'filename', 'full_path', 'zero_in_nanoseconds_lf', 'zero_in_nanoseconds_suspicious', 'zero_in_nanoseconds_combined', 'is_flagged_suspicious', 'ground_truth_label']


## Cell 4: Extract Forensic Pattern Features (15 features)

Based on Oh et al. timestomping indicators


In [4]:
# Cell 4: Extract Forensic Pattern Features

print("\n" + "=" * 80)
print("EXTRACTING FORENSIC PATTERN FEATURES")
print("=" * 80)

# Feature 1: Zero nanoseconds in LogFile (from lf_detail field)
print("\n1. Zero in nanoseconds (LogFile)")
if 'lf_detail' in df.columns:
    df['zero_in_nanoseconds_lf'] = df['lf_detail'].fillna('').str.contains(
        'Zero in 100-nanoseconds', case=False, na=False
    )
    print(f"   Detected: {df['zero_in_nanoseconds_lf'].sum()} files")
else:
    df['zero_in_nanoseconds_lf'] = False
    print("   Column 'lf_detail' not found, set to False")

# Feature 2: Zero nanoseconds combined (LogFile only for production)
print("\n2. Zero in nanoseconds (Combined)")
df['zero_in_nanoseconds_combined'] = df['zero_in_nanoseconds_lf']
print(f"   Detected: {df['zero_in_nanoseconds_combined'].sum()} files")

# Feature 3: Time reversal event
print("\n3. Time reversal event")
if 'lf_event' in df.columns:
    df['time_reversal_event'] = df['lf_event'].fillna('').str.contains(
        'Time Reversal', case=False, na=False
    )
    print(f"   Detected: {df['time_reversal_event'].sum()} files")
else:
    df['time_reversal_event'] = False
    print("   Column 'lf_event' not found, set to False")

# Feature 4: Timestamp changed to past
print("\n4. Timestamp changed to past")
if 'lf_detail' in df.columns:
    df['timestamp_changed_to_past'] = df['lf_detail'].fillna('').str.contains(
        'changed to the past', case=False, na=False
    )
    print(f"   Detected: {df['timestamp_changed_to_past'].sum()} files")
else:
    df['timestamp_changed_to_past'] = False
    print("   Column 'lf_detail' not found, set to False")

# Features 5-8: Timestamp modification patterns (CreationTime, ModifiedTime, MFTModifiedTime, AccessedTime)
print("\n5-8. Timestamp modification patterns")
timestamp_types = ['CreationTime', 'ModifiedTime', 'MFTModifiedTime', 'AccessedTime']
for ts_type in timestamp_types:
    feature_name = f'modified_{ts_type.lower()}'
    if 'lf_detail' in df.columns:
        df[feature_name] = df['lf_detail'].fillna('').str.contains(
            ts_type, case=False, na=False
        )
        print(f"   {feature_name}: {df[feature_name].sum()} files")
    else:
        df[feature_name] = False

# Feature 9: Using another file's timestamp
print("\n9. Using another file's timestamp")
if 'lf_detail' in df.columns:
    df['using_another_timestamp'] = df['lf_detail'].fillna('').str.contains(
        'Using another', case=False, na=False
    )
    print(f"   Detected: {df['using_another_timestamp'].sum()} files")
else:
    df['using_another_timestamp'] = False
    print("   Column 'lf_detail' not found, set to False")

# Feature 10: Basic info changed (UsnJrnl)
print("\n10. Basic info changed")
if 'usn_event_info' in df.columns:
    df['basic_info_changed'] = df['usn_event_info'].fillna('').str.contains(
        'Basic_Info_Change', case=False, na=False
    )
    print(f"   Detected: {df['basic_info_changed'].sum()} files")
else:
    df['basic_info_changed'] = False
    print("   Column 'usn_event_info' not found, set to False")

# Feature 11: Update resident value
print("\n11. Update resident value")
if 'lf_event' in df.columns:
    df['update_resident_value'] = df['lf_event'].fillna('').str.contains(
        'Update Resident Value', case=False, na=False
    )
    print(f"   Detected: {df['update_resident_value'].sum()} files")
else:
    df['update_resident_value'] = False
    print("   Column 'lf_event' not found, set to False")

# Feature 12: SI timestamp changed
print("\n12. SI timestamp changed")
if 'lf_detail' in df.columns:
    df['si_timestamp_changed'] = df['lf_detail'].fillna('').str.contains(
        r'\$SI', case=False, na=False, regex=True
    )
    print(f"   Detected: {df['si_timestamp_changed'].sum()} files")
else:
    df['si_timestamp_changed'] = False
    print("   Column 'lf_detail' not found, set to False")

# Feature 13: FN timestamp changed
print("\n13. FN timestamp changed")
if 'lf_detail' in df.columns:
    df['fn_timestamp_changed'] = df['lf_detail'].fillna('').str.contains(
        r'\$FN', case=False, na=False, regex=True
    )
    print(f"   Detected: {df['fn_timestamp_changed'].sum()} files")
else:
    df['fn_timestamp_changed'] = False
    print("   Column 'lf_detail' not found, set to False")

# Feature 14-15: Refined detection patterns
print("\n14. Zero nano + time reversal (combined)")
df['zero_nano_time_reversal'] = df['zero_in_nanoseconds_combined'] & df['time_reversal_event']
print(f"   Detected: {df['zero_nano_time_reversal'].sum()} files")

print("\n15. Zero nano + basic info changed")
df['zero_nano_basic_info'] = df['zero_in_nanoseconds_combined'] & df['basic_info_changed']
print(f"   Detected: {df['zero_nano_basic_info'].sum()} files")

print("\nForensic pattern features extracted: 15 features")



EXTRACTING FORENSIC PATTERN FEATURES

1. Zero in nanoseconds (LogFile)
   Detected: 19 files

2. Zero in nanoseconds (Combined)
   Detected: 19 files

3. Time reversal event
   Detected: 135 files

4. Timestamp changed to past
   Detected: 0 files

5-8. Timestamp modification patterns
   modified_creationtime: 90 files
   modified_modifiedtime: 60 files
   modified_mftmodifiedtime: 51 files
   modified_accessedtime: 14 files

9. Using another file's timestamp
   Detected: 0 files

10. Basic info changed
   Detected: 3491 files

11. Update resident value
   Detected: 0 files

12. SI timestamp changed
   Detected: 0 files

13. FN timestamp changed
   Detected: 0 files

14. Zero nano + time reversal (combined)
   Detected: 19 files

15. Zero nano + basic info changed
   Detected: 19 files

Forensic pattern features extracted: 15 features


## Cell 5: Extract Cross-Artifact Validation Features (3 features)

Validate detections across LogFile and UsnJrnl


In [5]:
# Cell 5: Extract Cross-Artifact Validation Features

print("\n" + "=" * 80)
print("EXTRACTING CROSS-ARTIFACT VALIDATION FEATURES")
print("=" * 80)

# Features already exist from notebook 01, but let's verify
print("\n1. Has LogFile evidence")
if 'has_logfile_evidence' not in df.columns:
    df['has_logfile_evidence'] = df['lf_lsn'].notna() if 'lf_lsn' in df.columns else False
print(f"   Files with LogFile evidence: {df['has_logfile_evidence'].sum()}")

print("\n2. Has UsnJrnl evidence")
if 'has_usnjrnl_evidence' not in df.columns:
    df['has_usnjrnl_evidence'] = df['usn_usn'].notna() if 'usn_usn' in df.columns else False
print(f"   Files with UsnJrnl evidence: {df['has_usnjrnl_evidence'].sum()}")

# Feature: Cross-artifact validation score
print("\n3. Cross-artifact validation score")
def calculate_cross_artifact_score(row):
    score = 0.0
    
    # LogFile evidence (Time Reversal or Update events)
    if row.get('has_logfile_evidence', False):
        if row.get('time_reversal_event', False) or row.get('update_resident_value', False):
            score += 0.5
    
    # UsnJrnl evidence (Basic_Info_Change)
    if row.get('has_usnjrnl_evidence', False):
        if row.get('basic_info_changed', False):
            score += 0.5
    
    return score

df['cross_artifact_validation_score'] = df.apply(calculate_cross_artifact_score, axis=1)

print(f"   Score distribution:")
print(df['cross_artifact_validation_score'].value_counts().sort_index())

print("\nCross-artifact validation features extracted: 3 features")



EXTRACTING CROSS-ARTIFACT VALIDATION FEATURES

1. Has LogFile evidence
   Files with LogFile evidence: 135

2. Has UsnJrnl evidence
   Files with UsnJrnl evidence: 3491

3. Cross-artifact validation score
   Score distribution:
cross_artifact_validation_score
0.5    3362
1.0     132
Name: count, dtype: int64

Cross-artifact validation features extracted: 3 features


## Cell 6: Extract File Characteristic Features (9 features)

File type, path, and location indicators


In [6]:
# Cell 6: Extract File Characteristic Features

print("\n" + "=" * 80)
print("EXTRACTING FILE CHARACTERISTIC FEATURES")
print("=" * 80)

# Get filename for analysis (use unified filename column)
filenames = df['filename'].fillna('')

# Feature 1: Is executable
print("\n1. Is executable")
df['is_executable'] = filenames.str.endswith(('.exe', '.dll', '.sys', '.bat', '.cmd', '.ps1'), na=False)
print(f"   Executables: {df['is_executable'].sum()} files")

# Feature 2: Is archive
print("\n2. Is archive")
df['is_archive'] = filenames.str.endswith(('.zip', '.rar', '.7z', '.tar', '.gz', '.bz2'), na=False)
print(f"   Archives: {df['is_archive'].sum()} files")

# Feature 3: Is document
print("\n3. Is document")
df['is_document'] = filenames.str.endswith(('.doc', '.docx', '.pdf', '.xls', '.xlsx', '.ppt', '.pptx', '.txt'), na=False)
print(f"   Documents: {df['is_document'].sum()} files")

# Feature 4: Is script
print("\n4. Is script")
df['is_script'] = filenames.str.endswith(('.vbs', '.js', '.py', '.sh', '.bash', '.pl'), na=False)
print(f"   Scripts: {df['is_script'].sum()} files")

# Feature 5: Is system file
print("\n5. Is system file")
df['is_system_file'] = filenames.str.endswith(('.sys', '.dll', '.drv'), na=False)
print(f"   System files: {df['is_system_file'].sum()} files")

# Feature 6: Path depth
print("\n6. Path depth")
paths = df['full_path'].fillna('')
df['path_depth'] = paths.str.count(r'[/\\\\]')
print(f"   Average path depth: {df['path_depth'].mean():.2f}")

# Feature 7: Filename length
print("\n7. Filename length")
df['filename_length'] = filenames.str.len()
print(f"   Average filename length: {df['filename_length'].mean():.2f}")

# Feature 8: Suspicious location
print("\n8. Suspicious location")
suspicious_paths = [
    'temp', 'tmp', 'appdata', 'programdata', 'public',
    'downloads', 'desktop', 'recycler', 'recycle'
]
pattern = '|'.join(suspicious_paths)
df['suspicious_location'] = paths.str.contains(pattern, case=False, na=False, regex=True)
print(f"   Suspicious locations: {df['suspicious_location'].sum()} files")

# Feature 9: Has file extension
print("\n9. Has file extension")
df['has_file_extension'] = filenames.str.contains(r'\.', na=False, regex=True)
print(f"   Files with extension: {df['has_file_extension'].sum()}")

print("\nFile characteristic features extracted: 9 features")



EXTRACTING FILE CHARACTERISTIC FEATURES

1. Is executable
   Executables: 178 files

2. Is archive
   Archives: 0 files

3. Is document
   Documents: 12 files

4. Is script
   Scripts: 24 files

5. Is system file
   System files: 161 files

6. Path depth
   Average path depth: 6.71

7. Filename length
   Average filename length: 37.12

8. Suspicious location
   Suspicious locations: 3044 files

9. Has file extension
   Files with extension: 3393

File characteristic features extracted: 9 features


## Cell 7: Extract Timestamp Relationship Features (2 features)

Analyze timestamp patterns and relationships


In [7]:
# Cell 7: Extract Timestamp Relationship Features

print("\n" + "=" * 80)
print("EXTRACTING TIMESTAMP RELATIONSHIP FEATURES")
print("=" * 80)

# Feature 1: Creation equals modified
print("\n1. Creation time equals modified time")
if 'lf_creation_time' in df.columns and 'lf_modified_time' in df.columns:
    df['creation_equals_modified'] = df['lf_creation_time'] == df['lf_modified_time']
    print(f"   Files where creation = modified: {df['creation_equals_modified'].sum()}")
elif 'usn_creation_time' in df.columns and 'usn_modified_time' in df.columns:
    df['creation_equals_modified'] = df['usn_creation_time'] == df['usn_modified_time']
    print(f"   Files where creation = modified: {df['creation_equals_modified'].sum()}")
else:
    df['creation_equals_modified'] = False
    print("   Timestamp columns not available, set to False")

# Feature 2: Timestamp equality (all 4 timestamps equal - highly suspicious)
print("\n2. All timestamps equal")
if all(col in df.columns for col in ['lf_creation_time', 'lf_modified_time', 'lf_mft_modified_time', 'lf_accessed_time']):
    df['timestamp_equality'] = (
        (df['lf_creation_time'] == df['lf_modified_time']) &
        (df['lf_modified_time'] == df['lf_mft_modified_time']) &
        (df['lf_mft_modified_time'] == df['lf_accessed_time'])
    )
    print(f"   Files with all timestamps equal: {df['timestamp_equality'].sum()}")
else:
    df['timestamp_equality'] = False
    print("   Timestamp columns not available, set to False")

print("\nTimestamp relationship features extracted: 2 features")



EXTRACTING TIMESTAMP RELATIONSHIP FEATURES

1. Creation time equals modified time
   Files where creation = modified: 0

2. All timestamps equal
   Files with all timestamps equal: 0

Timestamp relationship features extracted: 2 features


## Cell 8: Feature Summary and Validation


In [8]:
# Cell 8: Feature Summary and Validation

print("\n" + "=" * 80)
print("FEATURE EXTRACTION SUMMARY")
print("=" * 80)

# Count feature columns (exclude original raw columns)
original_cols = df.columns.tolist()
feature_cols = [col for col in original_cols if not col.startswith(('lf_', 'usn_', 'merge_key', 'filename', 'full_path'))]

print(f"\nTotal features extracted: {len(feature_cols)}")
print(f"Total columns in dataset: {len(df.columns)}")

# Display feature categories
print("\nFeature categories:")
print("  Forensic patterns: 15")
print("  Cross-artifact validation: 3")
print("  File characteristics: 9")
print("  Timestamp relationships: 2")
print("  Refined detection: 2")
print("  " + "-" * 40)
print("  Total: 31 features")

# Check for any features with all False/zero values
print("\nFeature validation:")
inactive_features = []
for col in feature_cols:
    if df[col].dtype == 'bool':
        if df[col].sum() == 0:
            inactive_features.append(col)
    elif df[col].dtype in ['int64', 'float64']:
        if df[col].sum() == 0:
            inactive_features.append(col)

if inactive_features:
    print(f"  WARNING: {len(inactive_features)} features have no detections:")
    for feat in inactive_features:
        print(f"    - {feat}")
else:
    print("  All features have non-zero values")

# Display top features by detection count
print("\nTop 10 features by detection count:")
feature_counts = {}
for col in feature_cols:
    if df[col].dtype == 'bool':
        feature_counts[col] = df[col].sum()
    elif col == 'cross_artifact_validation_score':
        feature_counts[col] = (df[col] > 0).sum()

top_features = sorted(feature_counts.items(), key=lambda x: x[1], reverse=True)[:10]
for feat, count in top_features:
    pct = (count / len(df)) * 100
    print(f"  {feat:40s}: {count:6,} ({pct:5.2f}%)")



FEATURE EXTRACTION SUMMARY

Total features extracted: 45
Total columns in dataset: 66

Feature categories:
  Forensic patterns: 15
  Cross-artifact validation: 3
  File characteristics: 9
  Timestamp relationships: 2
  Refined detection: 2
  ----------------------------------------
  Total: 31 features

Feature validation:
    - Carving Flag
    - timestamp_changed_to_past
    - using_another_timestamp
    - update_resident_value
    - si_timestamp_changed
    - fn_timestamp_changed
    - is_archive
    - creation_equals_modified
    - timestamp_equality

Top 10 features by detection count:
  cross_artifact_validation_score         :  3,494 (100.00%)
  has_usnjrnl_evidence                    :  3,491 (99.91%)
  basic_info_changed                      :  3,491 (99.91%)
  has_file_extension                      :  3,393 (97.11%)
  suspicious_location                     :  3,044 (87.12%)
  is_executable                           :    178 ( 5.09%)
  is_system_file                        

## Cell 9: Save Feature Dataset


In [9]:
# Cell 9: Save Feature Dataset

print("\n" + "=" * 80)
print("SAVING FEATURE DATASET")
print("=" * 80)

# Save to CSV
df.to_csv(output_path, index=False)

# Verify save
file_size_mb = os.path.getsize(output_path) / 1024 / 1024

print(f"\nDataset saved successfully")
print(f"  Output file: {output_path}")
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Features: 31")
print(f"  File size: {file_size_mb:.2f} MB")

print("\n" + "=" * 80)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 80)
print("\nNext step: Open '03 - Run Detection.ipynb' to run LightGBM model predictions")



SAVING FEATURE DATASET

Dataset saved successfully
  Output file: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Post-Processing Filter/12-KSK/data_features.csv
  Records: 3,494
  Columns: 66
  Features: 31
  File size: 2.19 MB

FEATURE ENGINEERING COMPLETE

Next step: Open '03 - Run Detection.ipynb' to run LightGBM model predictions
